# SCENIC+ pipeline

For this tutorial, we are processing the medulloblastoma mouse model single cell multiome data from:

[_Shiraishi, R. & Cancila, G. et al. (2024). Cancer-specific epigenome identifies oncogenic hijacking by nuclear factor I family proteins for medulloblastoma progression. Dev. Cell , 59:2302-2319._](https://www.cell.com/developmental-cell/fulltext/S1534-5807(24)00330-7)

This data set contains three samples comprising FACS sorted cells. These cells reflect the progression from healthy precursors to tumor cells:

1. Ptch1GNP: granule neuron precursors, P7
2. PNC: preneoplastic cells, P28
3. Tumor: tumor cells, adult mice

Data was downloaded from the Gene Expression Omnibus:

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE240362

The `GSE240362_RAW.tar` file contains the raw gene expression counts files in HDF5 format (.h5) and ATAC fragment files. This is the typical output you would obtain from the [10X Genomics Cellranger software](https://www.10xgenomics.com/support/software/cell-ranger/latest/getting-started/cr-what-is-cell-ranger).

The ATAC modality (fragment files) were processed with _pycisTopic_ and gene expression (.5) was processed with _Scanpy_. In addition, _cisTarget_ transcription factor binding site motif databases were generated. The SCENIC+ pipelines was then used to infer eRegulon. To see how the processing was done, see the following notebooks:

* [scanpy_rna_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/scanpy_rna_processing.ipynp)
* [pycistopic_atac_processing.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistopic_atac_processing.ipynp)
* [pycistarget_tfmotif_database.ipynp](https://github.com/heckern/2026_ebi_workshop_scenicplus/pycistarget_tfmotif_database.ipynp)

For more details on SCENIC+, have a look at:

* [_Bravo González-Blas, C., De Winter, S., Hulselmans, G., Hecker, N., Matetovici, I., Christiaens, V., ... & Aerts, S. (2023). SCENIC+: single-cell multiomic inference of enhancers and gene regulatory networks. Nat. Methods, 20:1355-1367._](https://www.nature.com/articles/s41592-023-01938-4)
* [SCENIC+ documentation](https://scenicplus.readthedocs.io/en/latest/)

This notebooks shows how the SCENIC+ pipeline was run.

## Configuration

The SCENIC+ pipeline is a Snakemake pipeline that uses a configuration file in YAML format as input.

In [1]:
import os

parentdir = '/storage/nhecker/embl/GSE240362/scenicplus_mm10'

os.makedirs(parentdir, exist_ok = True)

The `init_snakemake` command creates the Snakemake folder structure and initial `config.yaml` file.

In [2]:
! export PATH=$PATH:/home/nikolai/.conda/envs/scenicplus/bin; scenicplus init_snakemake --out_dir /storage/nhecker/embl/GSE240362/scenicplus_mm10

/home/nikolai/.conda/envs/scenicplus/lib/python3.11/site-packages/scenicplus/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2026-02-17 18:00:52,300 SCENIC+      INFO     Creating snakemake folder in: /storage/nhecker/embl/GSE240362/scenicplus_mm10


We create a helper function to view the files in the folder.

In [3]:
import os

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print('{}{}'.format(subindent, f))

In [4]:
list_files('/storage/nhecker/embl/GSE240362/scenicplus_mm10')

scenicplus_mm10/
    Snakemake/
        workflow/
            Snakefile
        config/
            config.yaml


We create an output and a temporary directory that will be used by the SCENIC+ pipeline.

In [5]:
tempdir = '/storage/nhecker/embl/GSE240362/scenicplus_mm10/temp'
outdir = '/storage/nhecker/embl/GSE240362/scenicplus_mm10/out'

os.makedirs(tempdir, exist_ok = True)
os.makedirs(outdir, exist_ok = True)

As a convenient way to modify the config file, we read it into dictionary using the `yaml` package.

In [6]:
import yaml

path_config = "/storage/nhecker/embl/GSE240362/scenicplus_mm10/Snakemake/config/config.yaml"

with open(path_config) as stream:
    dict_config = yaml.safe_load(stream)

In [7]:
dict_config.keys()

dict_keys(['input_data', 'output_data', 'params_general', 'params_data_preparation', 'params_motif_enrichment', 'params_inference'])

We have to specify the cisTopic object, gene expression anndata object, region set folders, pycisTarget database files, and motif annotation files as input for the pipeline.

In [8]:
dict_config['input_data']

{'cisTopic_obj_fname': '',
 'GEX_anndata_fname': '',
 'region_set_folder': '',
 'ctx_db_fname': '',
 'dem_db_fname': '',
 'path_to_motif_annotations': ''}

In [9]:
dict_config['input_data'] = {
    'cisTopic_obj_fname': '/storage/nhecker/embl/GSE240362/pycistopic_mm10/cistopic_obj.pkl',
    'GEX_anndata_fname': '/storage/nhecker/embl/GSE240362/scanpy/rna_medulloblastoma.h5ad',
    'region_set_folder': '/storage/nhecker/embl/GSE240362/pycistopic_mm10/region_sets',
    'ctx_db_fname': '/storage/nhecker/embl/GSE240362/cistargetdb_mm10/GSE240362_1kb_bg_with_mask.regions_vs_motifs.rankings.feather',
    'dem_db_fname': '/storage/nhecker/embl/GSE240362/cistargetdb_mm10/GSE240362_1kb_bg_with_mask.regions_vs_motifs.scores.feather',
    'path_to_motif_annotations': '/storage/nhecker/resources/cistarget_motif_db/v10nr_clust_public/snapshots/motifs-v10-nr.mgi-m0.00001-o0.0.tbl'
}

Additional config parameters that might have to be adjusted is the number of CPUs and the temp directory.

In [10]:
dict_config['params_general']

{'temp_dir': '', 'n_cpu': 40, 'seed': 666}

In [11]:
tempdir = '/storage/nhecker/embl/GSE240362/scenicplus_mm10/temp'

dict_config['params_general']['n_cpu'] = 32
dict_config['params_general']['temp_dir'] = tempdir

As we are using an older version the mouse genome (mm10). We have to make sure to specify a matching ENSEMBL biomart host for this genome version. This can be changed in the `params_data_preparation` section.

In [12]:
dict_config['params_data_preparation']

{'bc_transform_func': '"lambda x: f\'{x}\'"',
 'is_multiome': True,
 'key_to_group_by': '',
 'nr_cells_per_metacells': 10,
 'direct_annotation': 'Direct_annot',
 'extended_annotation': 'Orthology_annot',
 'species': 'hsapiens',
 'biomart_host': 'http://www.ensembl.org',
 'search_space_upstream': '1000 150000',
 'search_space_downstream': '1000 150000',
 'search_space_extend_tss': '10 10'}

The search space space specifies the distance to between transcription start site (TSS) and potential enhancer regions that SCENIC+ should consider for linking regions to genes.

In [13]:
dict_config['params_data_preparation']['biomart_host'] = 'http://aug2020.archive.ensembl.org'

The `params_motif_enrichment` contains parameters related to selecting regions and predicted motifs to be considered by SCENIC+.

In [14]:
dict_config['params_motif_enrichment']

{'species': 'homo_sapiens',
 'annotation_version': 'v10nr_clust',
 'motif_similarity_fdr': 0.001,
 'orthologous_identity_threshold': 0.0,
 'annotations_to_use': 'Direct_annot Orthology_annot',
 'fraction_overlap_w_dem_database': 0.4,
 'dem_max_bg_regions': 500,
 'dem_balance_number_of_promoters': True,
 'dem_promoter_space': 1000,
 'dem_adj_pval_thr': 0.05,
 'dem_log2fc_thr': 1.0,
 'dem_mean_fg_thr': 0.0,
 'dem_motif_hit_thr': 3.0,
 'fraction_overlap_w_ctx_database': 0.4,
 'ctx_auc_threshold': 0.005,
 'ctx_nes_threshold': 3.0,
 'ctx_rank_threshold': 0.05}

Potentially, the `dem_adj_pval_thr` has to be relaxed in the case the pipeline does not successfully complete due to the lack of enriched regions.

In [15]:
dict_config['params_motif_enrichment']['dem_adj_pval_thr'] = 0.5

We change the species here form human to mouse.

In [16]:
dict_config['params_motif_enrichment']['species'] = 'mus_musculus'

In [17]:
dict_config['params_data_preparation']['species'] = 'mmusculus'

For specifying output files, we append the output folder name to the default file names.

In [18]:
outdir = '/storage/nhecker/embl/GSE240362/scenicplus_mm10/out'

for key, file in dict_config['output_data'].items():
    dict_config['output_data'][key] = outdir + '/' + file

After modifying the values in the dictionary, we save it as YAML file.

In [19]:
path_config = "/storage/nhecker/embl/GSE240362/scenicplus_mm10/Snakemake/config/config.yaml"

with open(path_config, 'w') as file:
    yaml.dump(dict_config, file, default_flow_style=False)

In [20]:
! cat /storage/nhecker/embl/GSE240362/scenicplus_mm10/Snakemake/config/config.yaml

input_data:
  GEX_anndata_fname: /storage/nhecker/embl/GSE240362/scanpy/rna_medulloblastoma.h5ad
  cisTopic_obj_fname: /storage/nhecker/embl/GSE240362/pycistopic_mm10/cistopic_obj.pkl
  ctx_db_fname: /storage/nhecker/embl/GSE240362/cistargetdb_mm10/GSE240362_1kb_bg_with_mask.regions_vs_motifs.rankings.feather
  dem_db_fname: /storage/nhecker/embl/GSE240362/cistargetdb_mm10/GSE240362_1kb_bg_with_mask.regions_vs_motifs.scores.feather
  path_to_motif_annotations: /storage/nhecker/resources/cistarget_motif_db/v10nr_clust_public/snapshots/motifs-v10-nr.mgi-m0.00001-o0.0.tbl
  region_set_folder: /storage/nhecker/embl/GSE240362/pycistopic_mm10/region_sets
output_data:
  AUCell_direct: /storage/nhecker/embl/GSE240362/scenicplus_mm10/out/AUCell_direct.h5mu
  AUCell_extended: /storage/nhecker/embl/GSE240362/scenicplus_mm10/out/AUCell_extended.h5mu
  chromsizes: /storage/nhecker/embl/GSE240362/scenicplus_mm10/out/chromsizes.tsv
  cistromes_direct: /storage/nhecker/embl/GSE240362/scenicplus_mm10/o

## Preparing the genome and genome annotation

To avoid issues during running the pipelines, it is a good idea to prepare the genome annotation and chromosome sizes files in advance.

In [21]:
os.chdir(f"{parentdir}/Snakemake")

In [22]:
! pwd

/storage/nhecker/embl/GSE240362/scenicplus_mm10/Snakemake


We can use `download_genome_annotations` command to obtain the initial files.

In [23]:
! export PATH=$PATH:/home/nikolai/.conda/envs/scenicplus/bin; scenicplus prepare_data download_genome_annotations

/home/nikolai/.conda/envs/scenicplus/lib/python3.11/site-packages/scenicplus/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
usage: scenicplus prepare_data download_genome_annotations [-h] --species
                                                           SPECIES
                                                           --genome_annotation_out_fname
                                                           GENOME_ANNOTATION_OUT_FNAME
                                                           --chromsizes_out_fname
                                                           CHROMSIZES_OUT_FNAME
                                                           [--biomart_host BIOMART_HOST]
                               

Here we have to make sure to specify an ENSEMBL biomart host that matches the mm10 mouse genome assembly that we are using.

In [24]:
! export PATH=$PATH:/home/nikolai/.conda/envs/scenicplus/bin; scenicplus prepare_data download_genome_annotations  --biomart_host http://aug2020.archive.ensembl.org --species mmusculus --genome_annotation_out_fname  /storage/nhecker/embl/GSE240362/scenicplus_mm10/out/genome_annotation.tsv --chromsizes_out_fname /storage/nhecker/embl/GSE240362/scenicplus_mm10/out/chromsizes.tsv

/home/nikolai/.conda/envs/scenicplus/lib/python3.11/site-packages/scenicplus/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2026-02-17 18:01:03,657 Download gene annotation INFO     Using genome: GRCm38.p6
Could not find Id on https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=genome&term=GRCm38.p6
Returning gene annotation without subestting for assembled chromosomesand converting to UCSC style. Please make sure that the chromosome namesin the returned object match with the chromosome names in the scplus_obj.Chromosome sizes will not be returned
2026-02-17 18:01:04,075 SCENIC+      INFO     Chrosomome sizes was not found, please provide this information manually.
2026-02-17 18:01:04,076 SCENIC+      IN

Next, we convert the gene annotation into the UCSC chromosome style.

In [25]:
import pandas as pd

path_annotation = "/storage/nhecker/embl/GSE240362/scenicplus_mm10/out/genome_annotation.tsv"

dfanno = pd.read_csv(path_annotation, sep ="\t")
dfanno

,Chromosome,Start,End,Strand,Gene,Transcription_Start_Site,Transcript_type
0,MT,14145,15288,+,mt-Cytb,14145,protein_coding
1,MT,13552,14070,-,mt-Nd6,14070,protein_coding
2,MT,11742,13565,+,mt-Nd5,11742,protein_coding
3,MT,10167,11544,+,mt-Nd4,10167,protein_coding
4,MT,9877,10173,+,mt-Nd4l,9877,protein_coding
...,...,...,...,...,...,...,...
55594,CHR_WSB_EIJ_MMCHR11_CTG3,83144420,83148428,+,LT629154.2,83144420,protein_coding
55595,CHR_WSB_EIJ_MMCHR11_CTG3,83254598,83256993,-,LT629154.3,83256993,protein_coding
55596,CHR_WSB_EIJ_MMCHR11_CTG3,83233324,83245134,-,Slfn14,83245134,protein_coding
55597,CHR_WSB_EIJ_MMCHR11_CTG3,83150467,83174093,+,Slfn3,83150467,protein_coding


We specify a helper function to rename the chromosomes.

In [26]:
def rename_chr_to_ucsc(chrom):

    if chrom == 'MT':
        return 'chrM'
    elif chrom == 'X' or chrom == 'Y' or chrom.isdigit():
        return 'chr' + chrom
    else:
        return chrom

In [27]:
dfanno['Chromosome'] = [rename_chr_to_ucsc(chrom) for chrom in dfanno['Chromosome'] ]

In [28]:
import numpy as np

np.unique(dfanno['Chromosome'])

array(['CHR_CAST_EI_MMCHR11_CTG4', 'CHR_CAST_EI_MMCHR11_CTG5',
       'CHR_MG104_PATCH', 'CHR_MG117_PATCH', 'CHR_MG132_PATCH',
       'CHR_MG153_PATCH', 'CHR_MG171_PATCH', 'CHR_MG184_PATCH',
       'CHR_MG190_MG3751_PATCH', 'CHR_MG191_PATCH', 'CHR_MG209_PATCH',
       'CHR_MG3172_PATCH', 'CHR_MG3231_PATCH', 'CHR_MG3251_PATCH',
       'CHR_MG3490_PATCH', 'CHR_MG3496_PATCH', 'CHR_MG3530_PATCH',
       'CHR_MG3561_PATCH', 'CHR_MG3562_PATCH', 'CHR_MG3618_PATCH',
       'CHR_MG3627_PATCH', 'CHR_MG3648_PATCH', 'CHR_MG3656_PATCH',
       'CHR_MG3683_PATCH', 'CHR_MG3686_PATCH', 'CHR_MG3700_PATCH',
       'CHR_MG3714_PATCH', 'CHR_MG3829_PATCH', 'CHR_MG3833_MG4220_PATCH',
       'CHR_MG3836_PATCH', 'CHR_MG3999_PATCH', 'CHR_MG4136_PATCH',
       'CHR_MG4138_PATCH', 'CHR_MG4151_PATCH', 'CHR_MG4180_PATCH',
       'CHR_MG4200_PATCH', 'CHR_MG4211_PATCH', 'CHR_MG4212_PATCH',
       'CHR_MG4213_PATCH', 'CHR_MG4214_PATCH', 'CHR_MG4222_MG3908_PATCH',
       'CHR_MG4243_PATCH', 'CHR_MG4248_PATCH', 'CHR_MG

In [29]:
# save annotatiom
dfanno.to_csv(path_annotation, sep="\t")

To prepare the chromosome sizes files for SCENIC+, we have to add column names.

In [30]:
# create chrom sizes
import pandas as pd

path_chromsizes = "/storage/genomes/mm10/mm10.chrom.sizes"

df = pd.read_csv(path_chromsizes, header=None, sep ="\t")
df

,0,1
0,chr1,195471971
1,chr2,182113224
2,chrX,171031299
3,chr3,160039680
4,chr4,156508116
...,...,...
61,chrUn_GL456396,21240
62,chrUn_GL456368,20208
63,chrM,16299
64,chr4_JH584292_random,14945


In [31]:
chromsizes = pd.DataFrame(
    {
        'Chromosome': df[0],
        'Start': 1,
        'End': df[1]
    }
)

chromsizes

,Chromosome,Start,End
0,chr1,1,195471971
1,chr2,1,182113224
2,chrX,1,171031299
3,chr3,1,160039680
4,chr4,1,156508116
...,...,...,...
61,chrUn_GL456396,1,21240
62,chrUn_GL456368,1,20208
63,chrM,1,16299
64,chr4_JH584292_random,1,14945


In [32]:
# save to out folder
path_out_chromsizes = dict_config['output_data']['chromsizes']

#chromsizes.to_csv(path_out_chromsizes, sep="\t", index=False)
chromsizes.to_csv(path_out_chromsizes, sep="\t", index=False)

To run the Snakemake pipeline, we simple call the 'snakemake' command. Here we specify the number of CPUs and set the latency to 30 seconds to avoid timeouts for downloading files.

In [ ]:
! export PATH=$PATH:/home/nikolai/.conda/envs/scenicplus/bin; snakemake --cores 32  --latency-wait 30

Assuming unrestricted shared filesystem usage for local execution.
Building DAG of jobs...
Using shell: /usr/bin/bash
Provided cores: 32
Rules claiming more threads will be scaled down.
Job stats:
job                           count
--------------------------  -------
AUCell_direct                     1
AUCell_extended                   1
all                               1
eGRN_direct                       1
eGRN_extended                     1
get_search_space                  1
motif_enrichment_cistarget        1
motif_enrichment_dem              1
prepare_GEX_ACC_multiome          1
prepare_menr                      1
region_to_gene                    1
scplus_mudata                     1
tf_to_gene                        1
total                            13

Select jobs to execute...
Execute 1 jobs...

[Tue Feb 17 18:01:05 2026]
localrule motif_enrichment_cistarget:
    input: /storage/nhecker/embl/GSE240362/pycistopic_mm10/region_sets, /storage/nhecker/embl/GSE240362/cistargetdb_